# Phase-1 Smoke Recovery Notebook

这个版本的 notebook 不再假设三套 suite 都要从头开始跑。它的目标是：

- 优先复用已经完成的 `iid` 和 `v4_lite` suite。
- 自动检查 `clean` suite 是否完整。如果缺少 `block_evaluation.json` / `heldout_evaluation.json`，只补跑 clean。
- `v4_lite` 协议验证、rollout evaluation 和 proxy suite 注册都使用**显式解析出的 suite 名称**，不再依赖会漂移的 `${TAG}`。

这适合当前这种情况：大部分实验已经完成，但最后 proxy 组装失败，不希望从头再跑所有实验。

In [1]:
!nvidia-smi

Fri Apr 24 02:34:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")

PyTorch: 2.10.0+cu128
CUDA available: True
cuDNN version: 91002


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd drive/"MyDrive"/"Colab Notebooks"/"auvhamnode"/"g3_5_6"

/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6


In [5]:
%pip install -q torchdiffeq pandas

## 恢复策略

当前 notebook 采用下面的恢复策略：

1. 扫描 `checkpoints/` 下现有的 smoke suites。
2. 对每个 suite 检查：
   - 是否有 `runs.tsv`
   - 是否每个 run 都有 `best_model.pt`
   - 是否每个 run 都有 `block_evaluation.json` 和 `heldout_evaluation.json`
   - 是否每个 run 都已经有 rollout summary
3. `iid` 和 `v4_lite` 优先复用已经完整的 suite。
4. `clean` 如果不完整，只补跑 clean，并且修正为 `--noise-profile clean --noise-protocol auto`。
5. proxy suite 始终新建，但只链接已有 runs，不会复制大文件。

In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

os.environ["PYTHON_BIN"] = "python"
ROOT = Path.cwd()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_ROOT = ROOT / "checkpoints"
LOCAL_PROXY_ROOT = Path("/content") / "_proxy_suites"
DRIVE_PROXY_EXPORT_ROOT = CHECKPOINT_ROOT

print(f"ROOT={ROOT}")
print(f"DEVICE={DEVICE}")
print(f"LOCAL_PROXY_ROOT={LOCAL_PROXY_ROOT}")
print(f"DRIVE_PROXY_EXPORT_ROOT={DRIVE_PROXY_EXPORT_ROOT}")


DEBUG_LOG_ROOT = ROOT / "notebook" / "_debug_logs"


def run(cmd, cwd=ROOT, capture=False, label=None):
    env = os.environ.copy()
    if isinstance(cmd, str):
        printable = cmd
        shell = True
        payload = cmd
    else:
        payload = [str(part) for part in cmd]
        printable = shlex.join(payload)
        shell = False

    print(printable)

    if not capture:
        subprocess.run(payload, shell=shell, check=True, cwd=cwd, env=env)
        return

    completed = subprocess.run(
        payload,
        shell=shell,
        check=False,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="", file=sys.stderr)
    if completed.returncode != 0:
        DEBUG_LOG_ROOT.mkdir(parents=True, exist_ok=True)
        log_label = label or "command"
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        stdout_path = DEBUG_LOG_ROOT / f"{stamp}_{log_label}.stdout.log"
        stderr_path = DEBUG_LOG_ROOT / f"{stamp}_{log_label}.stderr.log"
        stdout_path.write_text(completed.stdout or "", encoding="utf-8")
        stderr_path.write_text(completed.stderr or "", encoding="utf-8")
        print("\n=== command failed ===")
        print(f"returncode: {completed.returncode}")
        print(f"command: {printable}")
        print(f"stdout log: {stdout_path}")
        print(f"stderr log: {stderr_path}")
        print("\n--- full stdout ---")
        print(completed.stdout or "<empty>")
        print("\n--- full stderr ---", file=sys.stderr)
        print(completed.stderr or "<empty>", file=sys.stderr)
        raise subprocess.CalledProcessError(
            completed.returncode,
            payload,
            output=completed.stdout,
            stderr=completed.stderr,
        )
    return completed


def _resolve_run_dir(suite_dir, run_dir_value):
    run_dir = Path(run_dir_value)
    if run_dir.is_absolute():
        return run_dir
    return (suite_dir / run_dir).resolve()


def _rollout_summary_count(run_dir):
    rollout_root = run_dir / "rollout_benchmark"
    if not rollout_root.exists():
        return 0
    return sum(1 for _ in rollout_root.rglob("summary.json"))


def _export_proxy_results(local_proxy_dir, export_dir, suite_names):
    export_dir.mkdir(parents=True, exist_ok=True)
    artifact_names = [
        "phase1_matrix.json",
        "phase1_summary.csv",
        "phase1_by_seed.csv",
        "phase1_by_scenario.csv",
        "phase1_degradation.csv",
        "experiment_report.md",
    ]
    copied = []
    for name in artifact_names:
        src = local_proxy_dir / name
        if src.exists():
            dst = export_dir / name
            shutil.copy2(src, dst)
            copied.append(dst.name)

    manifest_lines = [
        f"Local proxy suite: {local_proxy_dir}",
        f"Exported at: {datetime.now().isoformat(timespec='seconds')}",
        "",
        "Source suites:",
        *[f"- {suite_name}" for suite_name in suite_names],
        "",
        "Copied artifacts:",
        *[f"- {name}" for name in copied],
    ]
    (export_dir / "proxy_export_info.txt").write_text(
        "\n".join(manifest_lines) + "\n",
        encoding="utf-8",
    )
    return export_dir


def suite_health(suite_dir):
    suite_dir = Path(suite_dir)
    runs_path = suite_dir / "runs.tsv"
    if not runs_path.exists():
        return {
            "name": suite_dir.name,
            "path": str(suite_dir),
            "exists": suite_dir.exists(),
            "runs_tsv": False,
            "n_runs": 0,
            "best_model_count": 0,
            "block_eval_count": 0,
            "heldout_eval_count": 0,
            "rollout_ready_count": 0,
            "report_exists": False,
            "training_complete": False,
            "eval_complete": False,
        }

    runs = pd.read_csv(runs_path, sep="\t")
    best_model_count = 0
    block_eval_count = 0
    heldout_eval_count = 0
    rollout_ready_count = 0
    for _, row in runs.iterrows():
        run_dir = _resolve_run_dir(suite_dir, row["run_dir"])
        best_model_count += int((run_dir / "best_model.pt").exists())
        block_eval_count += int((run_dir / "block_evaluation.json").exists())
        heldout_eval_count += int((run_dir / "heldout_evaluation.json").exists())
        rollout_ready_count += int(_rollout_summary_count(run_dir) > 0)

    n_runs = len(runs)
    report_exists = (suite_dir / "experiment_report.md").exists()
    training_complete = (
        n_runs > 0
        and best_model_count == n_runs
        and block_eval_count == n_runs
        and heldout_eval_count == n_runs
    )
    eval_complete = training_complete and rollout_ready_count == n_runs and report_exists
    return {
        "name": suite_dir.name,
        "path": str(suite_dir),
        "exists": True,
        "runs_tsv": True,
        "n_runs": n_runs,
        "best_model_count": best_model_count,
        "block_eval_count": block_eval_count,
        "heldout_eval_count": heldout_eval_count,
        "rollout_ready_count": rollout_ready_count,
        "report_exists": report_exists,
        "training_complete": training_complete,
        "eval_complete": eval_complete,
    }


def candidate_suites(kind):
    patterns = {
        "clean": "sweep_oc_phase1_smoke_clean_*",
        "iid": "sweep_oc_phase1_smoke_iid_*",
        "v4lite": "sweep_oc_phase1_smoke_v4lite_*",
    }
    return sorted(CHECKPOINT_ROOT.glob(patterns[kind]), key=lambda path: path.name, reverse=True)


def find_best_suite(kind, require_eval=False):
    infos = [suite_health(path) for path in candidate_suites(kind)]
    for info in infos:
        if info["training_complete"] and (not require_eval or info["eval_complete"]):
            return info, infos
    return (infos[0] if infos else None), infos


def show_suite_infos(title, infos):
    print(title)
    if not infos:
        print("No candidate suites found.")
        return
    display(
        pd.DataFrame(infos)[[
            "name",
            "n_runs",
            "best_model_count",
            "block_eval_count",
            "heldout_eval_count",
            "rollout_ready_count",
            "report_exists",
            "training_complete",
            "eval_complete",
        ]]
    )

ROOT=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6
DEVICE=cuda
LOCAL_PROXY_ROOT=/content/_proxy_suites
DRIVE_PROXY_EXPORT_ROOT=/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints


In [ ]:
MODELS = ("phnode_full", "ablate_no_mass_prior")
SEEDS = (43, 44)
DATASET = "data/auv_oc_traj1000_blk150_s23_d0be9434.pkl"
SMOKE_TRAIN_ARGS = [
    "--batch_size", "512",
    "--epochs", "120",
    "--total_steps", "1500",
]

clean_pick, clean_infos = find_best_suite("clean", require_eval=False)
iid_pick, iid_infos = find_best_suite("iid", require_eval=True)
v4_pick, v4_infos = find_best_suite("v4lite", require_eval=True)

show_suite_infos("Clean candidates", clean_infos)
show_suite_infos("IID candidates", iid_infos)
show_suite_infos("V4-lite candidates", v4_infos)

CLEAN_SUITE = clean_pick["name"] if clean_pick is not None else None
IID_SUITE = iid_pick["name"] if iid_pick is not None else None
V4_SUITE = v4_pick["name"] if v4_pick is not None else None

NEED_CLEAN_RERUN = clean_pick is None or not clean_pick["training_complete"]

print(f"Initial CLEAN_SUITE={CLEAN_SUITE}")
print(f"Initial IID_SUITE={IID_SUITE}")
print(f"Initial V4_SUITE={V4_SUITE}")
print(f"NEED_CLEAN_RERUN={NEED_CLEAN_RERUN}")

assert IID_SUITE is not None, "No reusable iid suite found. If needed, re-run iid training manually."
assert V4_SUITE is not None, "No reusable v4_lite suite found. If needed, re-run v4 training manually."

Clean candidates


,name,n_runs,best_model_count,block_eval_count,heldout_eval_count,rollout_ready_count,report_exists,training_complete,eval_complete
0,sweep_oc_phase1_smoke_clean_fix_20260423_124711,4,4,4,4,4,True,True,True
1,sweep_oc_phase1_smoke_clean_colab_l4_smoke_202...,1,1,0,0,1,True,False,False


IID candidates


,name,n_runs,best_model_count,block_eval_count,heldout_eval_count,rollout_ready_count,report_exists,training_complete,eval_complete
0,sweep_oc_phase1_smoke_iid_colab_l4_smoke_20260...,4,4,4,4,4,True,True,True


V4-lite candidates


,name,n_runs,best_model_count,block_eval_count,heldout_eval_count,rollout_ready_count,report_exists,training_complete,eval_complete
0,sweep_oc_phase1_smoke_v4lite_colab_l4_smoke_20...,4,4,4,4,4,True,True,True


Initial CLEAN_SUITE=sweep_oc_phase1_smoke_clean_fix_20260423_124711
Initial IID_SUITE=sweep_oc_phase1_smoke_iid_colab_l4_smoke_20260423_0948
Initial V4_SUITE=sweep_oc_phase1_smoke_v4lite_colab_l4_smoke_20260423_0948
NEED_CLEAN_RERUN=False


## 只在必要时补跑 clean

这是整个恢复流程最重要的一步。当前代码下，如果 clean 训练用了 `--noise-protocol clean`，训练结束后的 noisy block/heldout eval 会失败。所以这里补跑 clean 时强制改成：

- `--noise-profile clean`
- `--noise-protocol auto`

这样 clean 训练本身仍然是 clean，但后续 noisy eval profile 能正常解析。

In [ ]:
def train_suite(
    suite_name,
    dataset,
    noise_profile,
    noise_protocol,
    models=MODELS,
    seeds=SEEDS,
    device=DEVICE,
    extra_train_args=None,
):
    cmd = [
        "bash", "scripts/train_all_models_noise_profile.sh",
        "--profile", "oc",
        "--models", " ".join(models),
        "--dataset", dataset,
        "--seeds", " ".join(map(str, seeds)),
        "--device", device,
        "--suite-name", suite_name,
        "--noise-profile", noise_profile,
        "--noise-protocol", noise_protocol,
        "--noise-reference", "remus100_dr",
        "--noise-warmup-epochs", "10",
        "--noise-ramp", "40",
        "--noise-mix-ratio", "0.5",
    ]
    for arg in extra_train_args or []:
        cmd += ["--extra-train-arg", str(arg)]
    run(cmd)
    return suite_name


FORCE_CLEAN_RERUN = False

if NEED_CLEAN_RERUN or FORCE_CLEAN_RERUN:
    CLEAN_SUITE = f"sweep_oc_phase1_smoke_clean_fix_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    print(f"Rerunning clean suite as: {CLEAN_SUITE}")
    train_suite(
        CLEAN_SUITE,
        dataset=DATASET,
        noise_profile="clean",
        noise_protocol="auto",
        extra_train_args=SMOKE_TRAIN_ARGS,
    )
else:
    print(f"Reusing existing clean suite: {CLEAN_SUITE}")

print(f"Resolved CLEAN_SUITE={CLEAN_SUITE}")
print(f"Resolved IID_SUITE={IID_SUITE}")
print(f"Resolved V4_SUITE={V4_SUITE}")

Rerunning clean suite as: sweep_oc_phase1_smoke_clean_fix_20260423_124711
bash scripts/train_all_models_noise_profile.sh --profile oc --models 'phnode_full ablate_no_mass_prior' --dataset data/auv_oc_traj1000_blk150_s23_d0be9434.pkl --seeds '43 44' --device cuda --suite-name sweep_oc_phase1_smoke_clean_fix_20260423_124711 --noise-profile clean --noise-protocol auto --noise-reference remus100_dr --noise-warmup-epochs 10 --noise-ramp 40 --noise-mix-ratio 0.5 --extra-train-arg --batch_size --extra-train-arg 512 --extra-train-arg --epochs --extra-train-arg 120 --extra-train-arg --total_steps --extra-train-arg 1500
Resolved CLEAN_SUITE=sweep_oc_phase1_smoke_clean_fix_20260423_124711
Resolved IID_SUITE=sweep_oc_phase1_smoke_iid_colab_l4_smoke_20260423_0948
Resolved V4_SUITE=sweep_oc_phase1_smoke_v4lite_colab_l4_smoke_20260423_0948


## 协议验证

这一步很轻量，但很关键。它不会重新训练，只会拿 `v4_lite` 的一个已有 run 做协议级检查。

In [ ]:
def first_run_dir(suite_name):
    runs = pd.read_csv(CHECKPOINT_ROOT / suite_name / "runs.tsv", sep="\t")
    return str(runs.iloc[0]["run_dir"])


V4_RUN_DIR = first_run_dir(V4_SUITE)
print(V4_RUN_DIR)

run([
    "python", "scripts/validate_v4_lite_protocol.py",
    "--run-dir", V4_RUN_DIR,
])

/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_smoke_v4lite_colab_l4_smoke_20260423_0948/main_phnode_full_seed43
python scripts/validate_v4_lite_protocol.py --run-dir '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_smoke_v4lite_colab_l4_smoke_20260423_0948/main_phnode_full_seed43'


## rollout evaluation

这里仍然对三套 suite 都调用 eval wrapper，但这不会盲目重跑所有 rollout：

- 已有 rollout 结果的 run 会被 wrapper 自动跳过。
- 缺失的结果会补上。
- 最后会重新生成 suite 级 summary / report，保证输出一致。

In [ ]:
def eval_suite(suite_name, eval_protocol, run_name):
    run([
        "bash", "scripts/eval_all_models_noise_profile.sh",
        "--suite-name", suite_name,
        "--device", DEVICE,
        "--mode", "resampled",
        "--num-traj-per-scenario", "8",
        "--times", "10 30 60",
        "--scenarios", "PRBS CHIRP OU",
        "--progress-every", "2",
        "--noise-profiles", "clean nominal_eval degraded_eval heading_biased_eval",
        "--noise-protocol", eval_protocol,
        "--noise-reference", "remus100_dr",
        "--noise-seed", "2024",
        "--extra-eval-arg", "--run_name",
        "--extra-eval-arg", run_name,
    ])


eval_suite(CLEAN_SUITE, "iid_noisy_ic", "resampled_traj8_seed42_iideval")
eval_suite(IID_SUITE, "iid_noisy_ic", "resampled_traj8_seed42_iideval")
eval_suite(V4_SUITE, "v4_lite", "resampled_traj8_seed42_v4eval")

bash scripts/eval_all_models_noise_profile.sh --suite-name sweep_oc_phase1_smoke_clean_fix_20260423_124711 --device cuda --mode resampled --num-traj-per-scenario 8 --times '10 30 60' --scenarios 'PRBS CHIRP OU' --progress-every 2 --noise-profiles 'clean nominal_eval degraded_eval heading_biased_eval' --noise-protocol iid_noisy_ic --noise-reference remus100_dr --noise-seed 2024 --extra-eval-arg --run_name --extra-eval-arg resampled_traj8_seed42_iideval
bash scripts/eval_all_models_noise_profile.sh --suite-name sweep_oc_phase1_smoke_iid_colab_l4_smoke_20260423_0948 --device cuda --mode resampled --num-traj-per-scenario 8 --times '10 30 60' --scenarios 'PRBS CHIRP OU' --progress-every 2 --noise-profiles 'clean nominal_eval degraded_eval heading_biased_eval' --noise-protocol iid_noisy_ic --noise-reference remus100_dr --noise-seed 2024 --extra-eval-arg --run_name --extra-eval-arg resampled_traj8_seed42_iideval
bash scripts/eval_all_models_noise_profile.sh --suite-name sweep_oc_phase1_smoke_

## 注册 proxy suite

proxy suite 是新的目录，但它只链接已有 run，不复制 checkpoint。这里不会再使用 `${TAG}` 去猜 suite 名称，而是直接使用上面已经解析出的 `CLEAN_SUITE / IID_SUITE / V4_SUITE`。

In [ ]:
# Use local /content directory for the proxy suite because Google Drive doesn't support symlinks
# Resolve absolute path for run_dir just in case it's relative in runs.tsv

def _suite_proxy_prefix(suite_name):
    lowered = str(suite_name).lower()
    if "v4lite" in lowered:
        return "v4lite"
    if "iid" in lowered:
        return "iid"
    if "clean" in lowered:
        return "clean"
    return lowered.replace("sweep_oc_", "").replace("/", "_")


def register_proxy_suite(proxy_suite_name, suite_names):
    local_proxy_dir = LOCAL_PROXY_ROOT / proxy_suite_name
    export_dir = DRIVE_PROXY_EXPORT_ROOT / proxy_suite_name
    cmd = [
        "python", "scripts/register_existing_runs_as_suite.py",
        "--suite-dir", str(local_proxy_dir),
        "--force",
    ]
    for suite_name in suite_names:
        suite_dir = CHECKPOINT_ROOT / suite_name
        prefix = _suite_proxy_prefix(suite_name)
        runs = pd.read_csv(suite_dir / "runs.tsv", sep="\t")
        for _, row in runs.iterrows():
            proxy_run_name = f"{prefix}__{row['run_name']}"
            source_run_dir = _resolve_run_dir(suite_dir, row["run_dir"])
            cmd += [
                "--run",
                str(row["group"]),
                str(row["model_type"]),
                str(int(row["seed"])),
                proxy_run_name,
                str(source_run_dir),
            ]
    run(cmd, capture=True, label="register_proxy_suite")
    run(["python", "scripts/summarize_sweep.py", "--suite-dir", str(local_proxy_dir)], capture=True, label="summarize_sweep")
    run(["python", "scripts/build_experiment_report.py", "--suite-dir", str(local_proxy_dir)], capture=True, label="build_experiment_report")
    exported_dir = _export_proxy_results(local_proxy_dir, export_dir, suite_names)
    return local_proxy_dir, exported_dir


PROXY_SUITE = f"sweep_oc_phase1_smoke_matched_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
try:
    proxy_local_dir, proxy_dir = register_proxy_suite(PROXY_SUITE, [CLEAN_SUITE, IID_SUITE, V4_SUITE])
    print(f"local proxy: {proxy_local_dir}")
    print(f"exported artifacts: {proxy_dir}")
except subprocess.CalledProcessError as exc:
    cmd_text = exc.cmd if isinstance(exc.cmd, str) else shlex.join(map(str, exc.cmd))
    print("\n[proxy stage failed]")
    print(f"returncode: {exc.returncode}")
    print(f"command: {cmd_text}")
    print("\n--- exception stdout ---")
    print(exc.output or "<empty>")
    print("\n--- exception stderr ---", file=sys.stderr)
    print(exc.stderr or "<empty>", file=sys.stderr)
    raise

python scripts/register_existing_runs_as_suite.py --suite-dir /content/_proxy_suites/sweep_oc_phase1_smoke_matched_20260423_173332 --force --run main phnode_full 43 clean__main_phnode_full_seed43 '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_smoke_clean_fix_20260423_124711/main_phnode_full_seed43' --run main phnode_full 44 clean__main_phnode_full_seed44 '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_smoke_clean_fix_20260423_124711/main_phnode_full_seed44' --run ablation ablate_no_mass_prior 43 clean__ablation_ablate_no_mass_prior_seed43 '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_smoke_clean_fix_20260423_124711/ablation_ablate_no_mass_prior_seed43' --run ablation ablate_no_mass_prior 44 clean__ablation_ablate_no_mass_prior_seed44 '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_smoke_clean_fix_20260423_124711/ablation_ablate_no_ma

## 查看结果

如果到这里都通过，说明：

- `v4_lite` 协议实现正常
- 三套 suite 至少已经处于可比较状态
- Phase-1 汇总合同和 report 生成链路都打通了

In [ ]:
summary = pd.read_csv(proxy_dir / "phase1_summary.csv")
degradation = pd.read_csv(proxy_dir / "phase1_degradation.csv")

print(proxy_dir / "phase1_summary.csv")
print(proxy_dir / "phase1_degradation.csv")
print(proxy_dir / "experiment_report.md")

display(summary.head(20))
display(degradation.head(20))

/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_smoke_matched_20260423_173332/phase1_summary.csv
/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_smoke_matched_20260423_173332/phase1_degradation.csv
/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_smoke_matched_20260423_173332/experiment_report.md


,group,model_type,train_noise_profile,train_noise_protocol,train_protocol_label,source,eval_profile,eval_protocol,eval_protocol_label,horizon_s,...,rollout_final_position_error_p95_min,rollout_final_position_error_p95_max,rollout_final_rotation_geodesic_median_mean,rollout_final_rotation_geodesic_median_std,rollout_final_rotation_geodesic_median_min,rollout_final_rotation_geodesic_median_max,rollout_final_total_linear_velocity_error_median_mean,rollout_final_total_linear_velocity_error_median_std,rollout_final_total_linear_velocity_error_median_min,rollout_final_total_linear_velocity_error_median_max
0,ablation,ablate_no_mass_prior,clean,clean,clean,block,clean,clean,clean,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ablation,ablate_no_mass_prior,clean,clean,clean,block,heading_biased_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ablation,ablate_no_mass_prior,clean,clean,clean,block,nominal_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ablation,ablate_no_mass_prior,clean,clean,clean,heldout,clean,clean,clean,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ablation,ablate_no_mass_prior,clean,clean,clean,heldout,degraded_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ablation,ablate_no_mass_prior,clean,clean,clean,heldout,heading_biased_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ablation,ablate_no_mass_prior,clean,clean,clean,heldout,nominal_eval,iid_noisy_ic,iid_noisy_ic,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ablation,ablate_no_mass_prior,clean,clean,clean,rollout,clean,clean,clean,10.0,...,1.561043,1.613698,0.039041,0.000643,0.038398,0.039684,0.084845,0.003928,0.080916,0.088773
8,ablation,ablate_no_mass_prior,clean,clean,clean,rollout,clean,clean,clean,30.0,...,5.399803,6.126029,0.048597,0.002188,0.046409,0.050785,0.070903,0.001687,0.069216,0.072589
9,ablation,ablate_no_mass_prior,clean,clean,clean,rollout,clean,clean,clean,60.0,...,11.166536,16.066873,0.073820,0.009076,0.064744,0.082896,0.077865,0.007192,0.070673,0.085057


,comparison_kind,suite_name,group,model_type,seed,run_name,run_dir,source,metric_name,horizon_s,...,baseline_train_protocol_label,eval_profile,baseline_eval_profile,eval_protocol,baseline_eval_protocol,value,clean_value,absolute_delta,ratio_to_clean,degradation_pct
0,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,block,block_angular_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.007034,0.007034,0.0,1.0,0.0
1,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,block,block_position_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.000475,0.000475,0.0,1.0,0.0
2,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,block,block_rotation_geodesic_mean,NaN,...,clean,clean,clean,clean,clean,0.000947,0.000947,0.0,1.0,0.0
3,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,block,block_velocity_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.005778,0.005778,0.0,1.0,0.0
4,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,heldout,heldout_angular_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.005828,0.005828,0.0,1.0,0.0
5,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,heldout,heldout_position_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.000416,0.000416,0.0,1.0,0.0
6,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,heldout,heldout_rotation_geodesic_mean,NaN,...,clean,clean,clean,clean,clean,0.000852,0.000852,0.0,1.0,0.0
7,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,heldout,heldout_success_rate,NaN,...,clean,clean,clean,clean,clean,1.000000,1.000000,0.0,1.0,0.0
8,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,heldout,heldout_velocity_rmse_mean,NaN,...,clean,clean,clean,clean,clean,0.004933,0.004933,0.0,1.0,0.0
9,clean_replay_cost,sweep_oc_phase1_smoke_matched_20260423_173332,ablation,ablate_no_mass_prior,43,iid__ablation_ablate_no_mass_prior_seed43,/content/_proxy_suites/sweep_oc_phase1_smoke_m...,rollout,rollout_completion_rate,10.0,...,clean,clean,clean,clean,clean,1.000000,1.000000,0.0,1.0,0.0


# 检查结果为什么一样

In [ ]:
import os
import json
import hashlib
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6")
CKPT = ROOT / "checkpoints"

CLEAN_SUITE = "sweep_oc_phase1_smoke_clean_fix_20260423_124711"
IID_SUITE = "sweep_oc_phase1_smoke_iid_colab_l4_smoke_20260423_0948"
V4_SUITE = "sweep_oc_phase1_smoke_v4lite_colab_l4_smoke_20260423_0948"

RUNS = [
    "main_phnode_full_seed43",
    "ablation_ablate_no_mass_prior_seed43",
    "main_phnode_full_seed44",
    "ablation_ablate_no_mass_prior_seed44",
]

KEYS = [
    "model_type",
    "seed",
    "noise_profile",
    "noise_protocol",
    "resolved_noise_profile",
    "resolved_noise_protocol",
    "noise_reference",
    "noise_mix_ratio",
    "noise_warmup_epochs",
    "noise_ramp_epochs",
]

def sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def inspect_run(suite_name, run_name):
    run_dir = CKPT / suite_name / run_name
    cfg = json.loads((run_dir / "config.json").read_text())
    result = {
        "suite": suite_name,
        "run": run_name,
        "config": {k: cfg.get(k) for k in KEYS},
        "best_model_sha256": sha256(run_dir / "best_model.pt"),
    }
    return result

for run_name in RUNS:
    print(f"\n===== {run_name} =====")
    rows = [
        inspect_run(CLEAN_SUITE, run_name),
        inspect_run(IID_SUITE, run_name),
        inspect_run(V4_SUITE, run_name),
    ]
    for row in rows:
        print(f"\n--- {row['suite']} ---")
        print(json.dumps(row["config"], indent=2, ensure_ascii=False))
        print("best_model_sha256:", row["best_model_sha256"])

    print("\nsha equalities:")
    print("clean == iid:", rows[0]["best_model_sha256"] == rows[1]["best_model_sha256"])
    print("clean == v4 :", rows[0]["best_model_sha256"] == rows[2]["best_model_sha256"])
    print("iid   == v4 :", rows[1]["best_model_sha256"] == rows[2]["best_model_sha256"])



===== main_phnode_full_seed43 =====

--- sweep_oc_phase1_smoke_clean_fix_20260423_124711 ---
{
  "model_type": "phnode_full",
  "seed": 43,
  "noise_profile": "clean",
  "noise_protocol": null,
  "resolved_noise_profile": "clean",
  "resolved_noise_protocol": "clean",
  "noise_reference": "remus100_dr",
  "noise_mix_ratio": 0.5,
  "noise_warmup_epochs": 10,
  "noise_ramp_epochs": 40
}
best_model_sha256: 393ae22db6ae46ca7cff516732794083dd4243322b83f727ae4856f73875e836

--- sweep_oc_phase1_smoke_iid_colab_l4_smoke_20260423_0948 ---
{
  "model_type": "phnode_full",
  "seed": 43,
  "noise_profile": "nominal_train",
  "noise_protocol": "iid_noisy_ic",
  "resolved_noise_profile": "nominal_train",
  "resolved_noise_protocol": "iid_noisy_ic",
  "noise_reference": "remus100_dr",
  "noise_mix_ratio": 0.5,
  "noise_warmup_epochs": 10,
  "noise_ramp_epochs": 40
}
best_model_sha256: 4289d7908ae69a24dd411801982ffea08e63c2c94f7f441fd4c1da2b781932d9

--- sweep_oc_phase1_smoke_v4lite_colab_l4_smoke_20

In [ ]:
import hashlib
from pathlib import Path

import torch

ROOT = Path("/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6")
CKPT = ROOT / "checkpoints"

CLEAN_SUITE = "sweep_oc_phase1_smoke_clean_fix_20260423_124711"
IID_SUITE = "sweep_oc_phase1_smoke_iid_colab_l4_smoke_20260423_0948"
V4_SUITE = "sweep_oc_phase1_smoke_v4lite_colab_l4_smoke_20260423_0948"

RUNS = [
    "main_phnode_full_seed43",
    "ablation_ablate_no_mass_prior_seed43",
    "main_phnode_full_seed44",
    "ablation_ablate_no_mass_prior_seed44",
]

def load_ckpt(suite_name, run_name):
    path = CKPT / suite_name / run_name / "best_model.pt"
    return torch.load(path, map_location="cpu", weights_only=False)

def state_fingerprint(state_dict):
    h = hashlib.sha256()
    for name, tensor in sorted(state_dict.items()):
        t = tensor.detach().cpu().contiguous()
        h.update(name.encode("utf-8"))
        h.update(str(tuple(t.shape)).encode("utf-8"))
        h.update(str(t.dtype).encode("utf-8"))
        h.update(t.numpy().tobytes())
    return h.hexdigest()

def compare_states(a, b):
    keys_a = set(a.keys())
    keys_b = set(b.keys())
    if keys_a != keys_b:
        return {
            "same_keys": False,
            "only_in_a": sorted(keys_a - keys_b),
            "only_in_b": sorted(keys_b - keys_a),
        }

    exact_equal = True
    max_abs_diff = 0.0
    sum_abs_diff = 0.0
    sum_sq_diff = 0.0
    sum_sq_ref = 0.0
    n = 0

    for k in sorted(keys_a):
        ta = a[k].detach().cpu()
        tb = b[k].detach().cpu()

        if not torch.equal(ta, tb):
            exact_equal = False

        da = (ta.to(torch.float64) - tb.to(torch.float64)).abs()
        if da.numel() > 0:
            max_abs_diff = max(max_abs_diff, da.max().item())
            sum_abs_diff += da.sum().item()
            sum_sq_diff += (da ** 2).sum().item()
            sum_sq_ref += (ta.to(torch.float64) ** 2).sum().item()
            n += da.numel()

    rel_l2 = (sum_sq_diff / max(sum_sq_ref, 1e-30)) ** 0.5
    mean_abs_diff = sum_abs_diff / max(n, 1)

    return {
        "same_keys": True,
        "exact_equal": exact_equal,
        "max_abs_diff": max_abs_diff,
        "mean_abs_diff": mean_abs_diff,
        "relative_l2_diff": rel_l2,
        "n_params": n,
    }

for run_name in RUNS:
    print(f"\n===== {run_name} =====")

    clean = load_ckpt(CLEAN_SUITE, run_name)
    iid = load_ckpt(IID_SUITE, run_name)
    v4 = load_ckpt(V4_SUITE, run_name)

    print("\ncheckpoint metadata:")
    for label, ckpt in [("clean", clean), ("iid", iid), ("v4", v4)]:
        print(
            f"{label:>5} | epoch={ckpt.get('epoch')} | loss={ckpt.get('loss'):.10f} | "
            f"state_fingerprint={state_fingerprint(ckpt['model_state_dict'])}"
        )

    print("\nmodel_state_dict diffs:")
    print("clean vs iid:", compare_states(clean["model_state_dict"], iid["model_state_dict"]))
    print("clean vs v4 :", compare_states(clean["model_state_dict"], v4["model_state_dict"]))
    print("iid   vs v4 :", compare_states(iid["model_state_dict"], v4["model_state_dict"]))



===== main_phnode_full_seed43 =====

checkpoint metadata:
clean | epoch=10 | loss=0.0556302845 | state_fingerprint=a6cadf2bcb72b5e608167a8d6297a343902b67343d6d2b7c0b2d3884b4a828d4
  iid | epoch=10 | loss=0.0556302845 | state_fingerprint=a6cadf2bcb72b5e608167a8d6297a343902b67343d6d2b7c0b2d3884b4a828d4
   v4 | epoch=10 | loss=0.0556302845 | state_fingerprint=a6cadf2bcb72b5e608167a8d6297a343902b67343d6d2b7c0b2d3884b4a828d4

model_state_dict diffs:
clean vs iid: {'same_keys': True, 'exact_equal': True, 'max_abs_diff': 0.0, 'mean_abs_diff': 0.0, 'relative_l2_diff': 0.0, 'n_params': 62806}
clean vs v4 : {'same_keys': True, 'exact_equal': True, 'max_abs_diff': 0.0, 'mean_abs_diff': 0.0, 'relative_l2_diff': 0.0, 'n_params': 62806}
iid   vs v4 : {'same_keys': True, 'exact_equal': True, 'max_abs_diff': 0.0, 'mean_abs_diff': 0.0, 'relative_l2_diff': 0.0, 'n_params': 62806}

===== ablation_ablate_no_mass_prior_seed43 =====

checkpoint metadata:
clean | epoch=10 | loss=0.0535097000 | state_fingerp

In [ ]:
from datetime import datetime

tag = datetime.now().strftime("%Y%m%d_%H%M%S")
DATASET = "/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl"
MODELS = "phnode_full ablate_no_mass_prior"
SEEDS = "43"

def train_probe_suite(suite_name, noise_profile, noise_protocol):
    run([
        "bash", "scripts/train_all_models_noise_profile.sh",
        "--profile", "oc",
        "--models", MODELS,
        "--dataset", DATASET,
        "--seeds", SEEDS,
        "--device", DEVICE,
        "--suite-name", suite_name,
        "--noise-profile", noise_profile,
        "--noise-protocol", noise_protocol,
        "--noise-reference", "remus100_dr",
        "--noise-warmup-epochs", "0",
        "--noise-ramp", "20",
        "--noise-mix-ratio", "1.0",
        "--extra-train-arg", "--batch_size",
        "--extra-train-arg", "512",
        "--extra-train-arg", "--epochs",
        "--extra-train-arg", "120",
        "--extra-train-arg", "--total_steps",
        "--extra-train-arg", "1500",
    ])

def eval_probe_suite(suite_name, eval_protocol, run_name):
    run([
        "bash", "scripts/eval_all_models_noise_profile.sh",
        "--suite-name", suite_name,
        "--device", DEVICE,
        "--mode", "resampled",
        "--num-traj-per-scenario", "8",
        "--times", "10 30 60",
        "--scenarios", "PRBS CHIRP OU",
        "--progress-every", "2",
        "--noise-profiles", "clean nominal_eval degraded_eval heading_biased_eval",
        "--noise-protocol", eval_protocol,
        "--noise-reference", "remus100_dr",
        "--noise-seed", "2024",
        "--extra-eval-arg", "--run_name",
        "--extra-eval-arg", run_name,
    ])

clean_suite = f"sweep_oc_phase1_probe_clean_{tag}"
iid_suite = f"sweep_oc_phase1_probe_iid_{tag}"
v4_suite = f"sweep_oc_phase1_probe_v4lite_{tag}"

train_probe_suite(clean_suite, "clean", "auto")
train_probe_suite(iid_suite, "nominal_train", "iid_noisy_ic")
train_probe_suite(v4_suite, "nominal_train", "v4_lite")

# 先做同一 eval protocol 的公平比较
eval_probe_suite(clean_suite, "iid_noisy_ic", "common_iideval")
eval_probe_suite(iid_suite, "iid_noisy_ic", "common_iideval")
eval_probe_suite(v4_suite, "iid_noisy_ic", "common_iideval")

proxy_local_dir, proxy_dir = register_proxy_suite(
    f"sweep_oc_phase1_probe_iideval_{tag}",
    [clean_suite, iid_suite, v4_suite],
)

print(proxy_dir)


bash scripts/train_all_models_noise_profile.sh --profile oc --models 'phnode_full ablate_no_mass_prior' --dataset '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl' --seeds 43 --device cuda --suite-name sweep_oc_phase1_probe_clean_20260423_180908 --noise-profile clean --noise-protocol auto --noise-reference remus100_dr --noise-warmup-epochs 0 --noise-ramp 20 --noise-mix-ratio 1.0 --extra-train-arg --batch_size --extra-train-arg 512 --extra-train-arg --epochs --extra-train-arg 120 --extra-train-arg --total_steps --extra-train-arg 1500
bash scripts/train_all_models_noise_profile.sh --profile oc --models 'phnode_full ablate_no_mass_prior' --dataset '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/data/auv_oc_traj1000_blk150_s23_d0be9434.pkl' --seeds 43 --device cuda --suite-name sweep_oc_phase1_probe_iid_20260423_180908 --noise-profile nominal_train --noise-protocol iid_noisy_ic --noise-reference remus100_dr --noise-warmup-epoc

NameError: name 'register_proxy_suite' is not defined

In [8]:
import os
import shlex
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

ROOT = Path("/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6")
CHECKPOINT_ROOT = ROOT / "checkpoints"
LOCAL_PROXY_ROOT = Path("/content") / "_proxy_suites"
DEBUG_LOG_ROOT = ROOT / "notebook" / "_debug_logs"

def run(cmd, cwd=ROOT, capture=False, label=None):
    env = os.environ.copy()
    if isinstance(cmd, str):
        printable = cmd
        shell = True
        payload = cmd
    else:
        payload = [str(part) for part in cmd]
        printable = shlex.join(payload)
        shell = False

    print(printable)

    if not capture:
        subprocess.run(payload, shell=shell, check=True, cwd=cwd, env=env)
        return

    completed = subprocess.run(
        payload,
        shell=shell,
        check=False,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="", file=sys.stderr)
    if completed.returncode != 0:
        DEBUG_LOG_ROOT.mkdir(parents=True, exist_ok=True)
        log_label = label or "command"
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        stdout_path = DEBUG_LOG_ROOT / f"{stamp}_{log_label}.stdout.log"
        stderr_path = DEBUG_LOG_ROOT / f"{stamp}_{log_label}.stderr.log"
        stdout_path.write_text(completed.stdout or "", encoding="utf-8")
        stderr_path.write_text(completed.stderr or "", encoding="utf-8")
        print(f"\n[command failed] returncode={completed.returncode}")
        print(f"stdout log: {stdout_path}")
        print(f"stderr log: {stderr_path}")
        raise subprocess.CalledProcessError(
            completed.returncode,
            payload,
            output=completed.stdout,
            stderr=completed.stderr,
        )
    return completed

def _resolve_run_dir(suite_dir, run_dir_value):
    run_dir = Path(run_dir_value)
    if run_dir.is_absolute():
        return run_dir
    return (suite_dir / run_dir).resolve()

def _suite_proxy_prefix(suite_name):
    lowered = str(suite_name).lower()
    if "v4lite" in lowered:
        return "v4lite"
    if "iid" in lowered:
        return "iid"
    if "clean" in lowered:
        return "clean"
    return lowered.replace("sweep_oc_", "").replace("/", "_")

def _export_proxy_results(local_proxy_dir, export_dir, suite_names):
    export_dir.mkdir(parents=True, exist_ok=True)
    artifact_names = [
        "phase1_matrix.json",
        "phase1_summary.csv",
        "phase1_by_seed.csv",
        "phase1_by_scenario.csv",
        "phase1_degradation.csv",
        "experiment_report.md",
    ]
    for name in artifact_names:
        src = local_proxy_dir / name
        if src.exists():
            shutil.copy2(src, export_dir / name)

def register_proxy_suite(proxy_suite_name, suite_names):
    local_proxy_dir = LOCAL_PROXY_ROOT / proxy_suite_name
    export_dir = CHECKPOINT_ROOT / proxy_suite_name

    cmd = [
        "python", "scripts/register_existing_runs_as_suite.py",
        "--suite-dir", str(local_proxy_dir),
        "--force",
    ]

    for suite_name in suite_names:
        suite_dir = CHECKPOINT_ROOT / suite_name
        runs = pd.read_csv(suite_dir / "runs.tsv", sep="\t")
        prefix = _suite_proxy_prefix(suite_name)
        for _, row in runs.iterrows():
            proxy_run_name = f"{prefix}__{row['run_name']}"
            source_run_dir = _resolve_run_dir(suite_dir, row["run_dir"])
            cmd += [
                "--run",
                str(row["group"]),
                str(row["model_type"]),
                str(int(row["seed"])),
                proxy_run_name,
                str(source_run_dir),
            ]

    run(cmd, capture=True, label="register_proxy_suite")
    run(
        ["python", "scripts/summarize_sweep.py", "--suite-dir", str(local_proxy_dir)],
        capture=True,
        label="summarize_sweep",
    )
    run(
        ["python", "scripts/build_experiment_report.py", "--suite-dir", str(local_proxy_dir)],
        capture=True,
        label="build_experiment_report",
    )
    _export_proxy_results(local_proxy_dir, export_dir, suite_names)
    return local_proxy_dir, export_dir

# 自动找你刚刚跑完的 probe suites
clean_candidates = sorted(CHECKPOINT_ROOT.glob("sweep_oc_phase1_probe_clean_*"))
iid_candidates = sorted(CHECKPOINT_ROOT.glob("sweep_oc_phase1_probe_iid_*"))
v4_candidates = sorted(CHECKPOINT_ROOT.glob("sweep_oc_phase1_probe_v4lite_*"))

assert clean_candidates, "No clean probe suite found"
assert iid_candidates, "No iid probe suite found"
assert v4_candidates, "No v4 probe suite found"

CLEAN_SUITE = clean_candidates[-1].name
IID_SUITE = iid_candidates[-1].name
V4_SUITE = v4_candidates[-1].name

print("CLEAN_SUITE =", CLEAN_SUITE)
print("IID_SUITE   =", IID_SUITE)
print("V4_SUITE    =", V4_SUITE)

PROXY_SUITE = f"sweep_oc_phase1_probe_iideval_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
proxy_local_dir, proxy_dir = register_proxy_suite(
    PROXY_SUITE,
    [CLEAN_SUITE, IID_SUITE, V4_SUITE],
)

print("local proxy:", proxy_local_dir)
print("exported artifacts:", proxy_dir)
print(proxy_dir / "phase1_summary.csv")
print(proxy_dir / "experiment_report.md")


CLEAN_SUITE = sweep_oc_phase1_probe_clean_20260423_180908
IID_SUITE   = sweep_oc_phase1_probe_iid_20260423_180908
V4_SUITE    = sweep_oc_phase1_probe_v4lite_20260423_180908
python scripts/register_existing_runs_as_suite.py --suite-dir /content/_proxy_suites/sweep_oc_phase1_probe_iideval_20260424_024232 --force --run main phnode_full 43 clean__main_phnode_full_seed43 '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_probe_clean_20260423_180908/main_phnode_full_seed43' --run ablation ablate_no_mass_prior 43 clean__ablation_ablate_no_mass_prior_seed43 '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_probe_clean_20260423_180908/ablation_ablate_no_mass_prior_seed43' --run main phnode_full 43 iid__main_phnode_full_seed43 '/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_6/checkpoints/sweep_oc_phase1_probe_iid_20260423_180908/main_phnode_full_seed43' --run ablation ablate_no_mass_prior 43 iid__ablation_ablate_no_ma